# Coding Agents

This notebook has two parts. 

**Part 1** teaches the concepts behind agentic coding tools — what an agent is, how permissions and planning keep it safe, and two ways you extend one with **skills** and **hooks**. We use **Claude Code** as the running example throughout, since it documents every one of these concepts clearly and by name.

**Part 2** is hands-on: you'll install **Cline**, a free VS Code extension, and connect it to a free API key (**Codestral** from Mistral, or **Gemini Flash** from Google) so you can actually use an agent today, without a paid subscription. 


For this course, you can choose to use a paid version for Claude Code or use a free agent with Cline.


## Part 1 — Concepts

### What is an agent?

> Claude Code is an agentic coding tool that reads your codebase, edits files, runs commands, and integrates with your development tools. Available in your terminal, IDE, desktop app, and browser.
>
> Claude Code is an AI-powered coding assistant that helps you build features, fix bugs, and automate development tasks. It understands your entire codebase and can work across multiple files and tools to get things done.

— [Claude Code docs, Overview](https://code.claude.com/docs/en/overview)

While Autocomplete suggests a line and Chat answers a question, an **agent** goes further as it reads your files, decides what to change, and *acts* on your codebase — editing files, running commands — often across several steps without you typing each one. 

That extra power is exactly why the rest of this notebook exists: once a tool can act on its own, *approval* (what it's allowed to do) and *undo* (what happens when it gets it wrong) become the interesting problems.


### The agentic loop

Every agentic coding tool runs the same loop:

```
   ┌─────────┐      ┌────────┐      ┌──────────┐
   │  PLAN   │ ───▶ │  ACT   │ ───▶ │ OBSERVE  │
   └─────────┘      └────────┘      └──────────┘
        ▲                                │
        └──────────── iterate ◀──────────┘
```

* *PLAN:* propose a step
* *ACT:* edit, file, run cmd
* *OBSERVE:* result (diff, output, errors)

The human operates on the **act**, and nothing happens until the human approves it (or the tool it's allowed to skip asking). Almost every feature covered below — permissions, plan mode, checkpoints, skills, hooks — is a control knob on one of these four arrows.


### Course policy

**Course policy**: starting **day 6**, you are allowed and encouraged to use coding agents for the exercises in the other notebooks. However, you are responsible for understanding and being able to explain any code your agent produces — in exams or oral evaluation you may be asked to justify a design choice, a line of code, or a bug fix that an agent wrote for you. "The agent did it" is not an acceptable answer.

The supported hands-on path for this course is **Cline in VS Code, driven by a free API key** (Part 2, below). **Claude Code** (or others) is permitted but requires a paid subscription you fund yourself.


### Permissions and approval

> Claude Code supports fine-grained permissions so that you can specify exactly what the agent is allowed to do and what it can't. You can check permission settings into version control to share them with every developer in your organization, and each developer can customize their own.
>
> * **Allow** rules let Claude Code use the specified tool without manual approval.
> * **Ask** rules prompt for confirmation whenever Claude Code tries to use the specified tool.
> * **Deny** rules prevent Claude Code from using the specified tool.

| Tool type | Example | Approval required |
|---|---|---|
| Read-only | File reads, Grep | No, within the working directory |
| Bash commands | Shell execution | Yes, except a built-in set of read-only commands |
| File modification | Edit/write files | Yes |

— [Claude Code docs, Configure permissions](https://code.claude.com/docs/en/permissions)

> Auto Approve lets you decide which actions Cline can take without prompting you each time.

Cline's categories: "Read project files", "Edit project files", "Execute safe commands", "Execute all commands", "Use the browser" — each toggled independently.

> YOLO mode is Auto Approve on steroids. Check the box and Cline auto-approves everything: file changes, terminal commands, browser actions, and mode transitions.
>
> Warning: This is dangerous. YOLO mode disables all safety checks.

— [Cline docs, Auto Approve](https://docs.cline.bot/features/auto-approve)

> `bypassPermissions` mode skips permission prompts, including for writes to `.git`, `.config/git`, `.claude`, `.vscode`, `.idea`, `.husky`, `.cargo`, `.devcontainer`, `.yarn`, and `.mvn`. Only use this mode in isolated environments like containers or VMs where Claude Code can't cause damage.

— [Claude Code docs, Configure permissions](https://code.claude.com/docs/en/permissions)

**This course does not endorse `bypassPermissions` / `--dangerously-skip-permissions` or Cline's YOLO mode** for the exercises: the whole point of the permission prompt is that a human looks at what's about to happen before it does. Auto-approving *reads* is fine — that's the read-only row in the table above. Auto-approving *writes* or *terminal commands* is not, for this course.


### Reading a diff before you approve

This is the single most important habit in this notebook. The checklist below it's the standard human code-review checklist, and an agent's diff deserves no less scrutiny:

> The most important thing to cover in a review is the overall design of the changelist (CL).
>
> Does this CL do what the developer intended?
>
> Is the CL more complex than it should be?
>
> Ask for unit, integration, or end-to-end tests as appropriate for the change.
>
> In the general case, look at *every* line of code that you have been assigned to review.

— Google Engineering Practices, [What to look for in a code review](https://google.github.io/eng-practices/review/reviewer/looking-for.html)

Applied to an agent's diff, before you click Approve:

1. **Design** — does it touch only the files/scope you asked for, or has it wandered?
2. **Functionality** — does it actually do what you asked, not just what the prose above it claims?
3. **Complexity** — did it reach for something more complex than the task needed (a new dependency, an unrequested abstraction)?
4. **Tests** — did it add, remove, or silently break test or output coverage?
5. **Every line** — can you explain every line, out loud, to someone else? (This doubles as the day-6 exam clause from the course policy above.)

Claude Code's own review skill draws the same "did it work" / "could it be better" line:

> `/code-review` reads the current diff and reports correctness bugs alongside reuse, simplification, and efficiency cleanups.

— [Claude Code docs, Code Review](https://code.claude.com/docs/en/code-review)

Two traps specific to notebooks, beyond either checklist above: an agent rewriting an *entire* cell when you asked for one comment, and an agent clobbering stored cell outputs it never should have touched.

### Planning vs. acting

Cline makes this the sharpest and most literal:

> Plan mode is where you and Cline figure out what you're building and how. In this mode, Cline can read your codebase, run searches, and discuss strategy, but cannot modify any files or execute commands.
>
> Once you have a plan, switch to Act mode. Cline retains the full context from your planning session and can now modify files, run commands, and execute your strategy.
>
> The conversation history carries over when you switch modes. Cline remembers everything you discussed in Plan mode, so you don't need to repeat yourself.

— [Cline docs, Plan & Act](https://docs.cline.bot/features/plan-and-act)

Claude Code names the same idea:

> `plan`: Claude reads files and runs read-only shell commands to explore but doesn't edit your source files; with auto mode available, classifier-approved commands also run. Labeled Plan in the CLI and the VS Code extension.

— [Claude Code docs, Configure permissions](https://code.claude.com/docs/en/permissions)

**Rule of thumb:** anything touching more than one file, or anything you're not 100% sure how to phrase precisely, starts in Plan. Read the plan with the same five-point checklist from above before switching to Act.


### Undo: checkpoints and git

Cline saves a snapshot of your project files every time it modifies one, and lets you restore code, conversation, or both to any earlier point — convenient, but it's a *feature of the tool*, not a guarantee.

The real undo is **git**: commit or stash *before* you let an agent write anything, and run `git diff` after, before you trust the result. That habit is what makes the exercises below low-risk regardless of which tool or checkpoint system you're using.


### Context and memory

> Each Claude Code session begins with a fresh context window. Two mechanisms carry knowledge across sessions:
> * **CLAUDE.md files**: instructions you write to give Claude persistent context
> * **Auto memory**: notes Claude writes itself based on your corrections and preferences

— [Claude Code docs, Memory](https://code.claude.com/docs/en/memory)

> Rules are markdown files that provide persistent instructions across all conversations. Instead of repeating the same preferences every time you start a new task, rules let you define them once and have Cline follow them automatically.
>
> Workspace rules go in `.clinerules/` at your project root. Use these for team standards, project-specific constraints, and anything you want to share with collaborators via version control.

— [Cline docs, Cline Rules](https://docs.cline.bot/customization/cline-rules)

Same concept, different filenames — a cross-tool map:

| Scope | Cline | Claude Code | Codex |
|---|---|---|---|
| Project (team-shared, in git) | `.clinerules/` | `./CLAUDE.md` | `AGENTS.md` |
| User (all your projects) | `~/Documents/Cline/Rules` | `~/.claude/CLAUDE.md` | `~/.agents/AGENTS.md` |
| Local (uncommitted) | *(toggle a rule off in the Rules panel)* | `./CLAUDE.local.md` | — |

Version-controlled prose that the agent reads every session is the cheapest lever you have — cheaper than any prompt you'll type by hand.


In [ ]:
from pathlib import Path
print(Path("CLAUDE.md").read_text())


### What a good rules file contains

The file printed above states facts and conventions (repo structure, dataset paths, "preserve pedagogical ordering") — not step-by-step procedures. That's the pattern to copy: durable truths about the project, not a to-do list.

> Run `/init` to generate a starting CLAUDE.md automatically. Claude analyzes your codebase and creates a file with build commands, test instructions, and project conventions it discovers. If a CLAUDE.md already exists, `/init` suggests improvements rather than overwriting it.

— [Claude Code docs, Memory](https://code.claude.com/docs/en/memory)

**Important gap:** Cline auto-detects `.clinerules/`, `.cursorrules`, `.windsurfrules`, and `AGENTS.md` — but it does **not** read `CLAUDE.md`. Without an `AGENTS.md`, Cline would start every session in this repo with zero context. 

This repository does not ship any context (`AGENTS.md` or `CLAUDE.md`)


### Managing the context window

Context is finite, and every turn re-sends everything in it — long sessions get slower and burn through your free-tier quota faster.

> `/clear`: Start a new conversation with empty context. [...] To free up context while continuing the same conversation, use `/compact` instead. [...] Aliases: `/reset`, `/new`
>
> `/compact`: Free up context by summarizing the conversation so far. Optionally pass focus instructions for the summary.
>
> `--continue`, `-c`: Load the most recent conversation in the current directory.

— [Claude Code docs, Commands reference](https://code.claude.com/docs/en/commands)

Cline's equivalent is simpler: there's no `/compact`, so start a **new task** per goal rather than piling unrelated work into one long conversation. One task, one goal — it's cheaper and the agent stays focused.


### Models and cost

> **`sonnet`**: Uses the latest Sonnet model for daily coding tasks
>
> **`opus`**: Uses the latest Opus model for complex reasoning tasks
>
> **`haiku`**: Uses the fast and efficient Haiku model for simple tasks

— [Claude Code docs, Model configuration](https://code.claude.com/docs/en/model-config)

> `/model`: Switch the AI model and save it as your default for new sessions.

— [Claude Code docs, Commands reference](https://code.claude.com/docs/en/commands)

The general axis is the same everywhere: fast-and-cheap vs. slow-and-strong. On the free path you'll set up in Part 2, `codestral-latest` and `gemini-2.5-flash` sit at the fast/cheap end — plenty for the exercises here. Two practical notes:

* Cline lets you set a **different model per mode** — a stronger model for Plan, a cheaper one for Act — if your quota allows it.
* Every retry re-sends the whole context, so a rambling, unfocused session burns through a free-tier quota far faster than a few short, targeted ones.


### Extending an agent: two mechanisms

An agent out of the box can only read and write files and run commands in your project. There are two distinct ways to give it more — remembering procedures, and guaranteeing side effects. They solve different problems, so it's worth keeping them separate.

### Skills — procedures the agent decides to use

> Skills extend what Claude can do. Create a `SKILL.md` file with instructions, and Claude adds it to its toolkit. Claude uses skills when relevant, or you can invoke one directly with `/skill-name`.
>
> Create a skill when you keep pasting the same instructions, checklist, or multi-step procedure into chat, or when a section of CLAUDE.md has grown into a procedure rather than a fact.

— [Claude Code docs, Skills](https://code.claude.com/docs/en/skills)

As an example, a minimal skill for explaining a notebook cell in plain language might look like this:

```markdown
---
name: explain-notebook-cell
description: Explain a Jupyter notebook code cell in plain, non-technical language, then suggest one short clarifying comment to add to it.
---

1. Read the target cell and the markdown immediately above it for context.
2. Summarize in 2-4 sentences what the cell does and why it matters, avoiding jargon.
3. Propose one short inline comment (max one line) that would help a student re-reading
   the cell later. Do not rewrite or restructure the code.
```

> Cline provides slash commands in chat that help you manage your conversation and plan complex implementations.

— [Cline docs, Slash Commands](https://docs.cline.bot/core-workflows/using-commands)

The same three-step procedure, saved as a Cline slash command (a markdown file under Cline's workflow settings, invoked as `/explain-notebook-cell` in chat), carries no fewer instructions — just a different file location and no YAML frontmatter. Same idea, two file formats: a **rule** (from the Context section above) is always loaded; a **skill** or **slash command** is loaded on demand.


### Hooks — guaranteed side effects

> Hooks are user-defined shell commands. Claude Code runs them at specific points in its lifecycle, which gives you deterministic control: certain actions always happen rather than relying on the LLM to choose to run them. Use hooks to enforce project rules, automate repetitive tasks, and integrate Claude Code with your existing tools.

— [Claude Code docs, Hooks guide](https://code.claude.com/docs/en/hooks-guide)

As an illustration of the shape of a hook configuration (in `settings.json`), a `PostToolUse` hook that auto-formats a Python file every time the agent edits one might look like this:

```json
{
  "hooks": {
    "PostToolUse": [
      {
        "matcher": "Edit",
        "hooks": [
          { "type": "command", "command": "black \"$CLAUDE_FILE_PATH\"" }
        ]
      }
    ]
  }
}
```

This is shown for illustration — you don't need to configure any hooks for this course.

**Cline has no equivalent.** This is one of the things a paid CLI agent adds. The closest Cline workaround is a rule that says "always run black after editing a Python file" — but that's a *request* to the model, not a *guarantee* like a hook. Knowing which of the two you're relying on matters: a rule can be forgotten or skipped under pressure; a hook cannot.

**The pair, side by side:** a skill or slash command reaches *sideways* (a saved procedure, run when needed); a hook reaches *downward* into the lifecycle (a shell command that always fires). Both are optional — neither is required to use an agent at all.


## Part 2 — Hands-on: Cline in VS Code

### Setup 1 — Install Cline

Open this repository **as a folder** in VS Code (if you normally work in JupyterLab, install the "Jupyter" extension too, so notebooks open and run the same way). 

1. Open the Extensions view: `Ctrl/Cmd + Shift + X`.
2. Search for "Cline" and click Install.
3. Open Cline from its icon in the sidebar.

![placeholder: the Cline icon in VS Code's activity bar / sidebar after installation, with the chat panel open](figures/cline-setup-sidebar-icon.png)

> Use this if you want Cline inside your editor UI.

— [Cline docs, Installing Cline](https://docs.cline.bot/getting-started/installing-cline)

Cline is a VS Code extension, so it also runs unmodified in VS Code forks (Cursor, Windsurf, VS Codium). 

After installing, complete provider setup in Cline's settings — that's the next two cells.

### Setup 2 — a free key: Codestral (Mistral)

1. Create or sign in to a Mistral platform account.
2. Generate an API key — from `codestral.mistral.ai` (Codestral-only key) or `api.mistral.ai` (a general "La Plateforme" key that also covers Codestral).

![placeholder: the Mistral platform page where you generate an API key (codestral.mistral.ai or api.mistral.ai)](figures/cline-setup-mistral-apikey.png)

3. In Cline, choose "Mistral" as the API Provider and paste the key. Pick `codestral-latest` as the model.

![placeholder: Cline's settings panel with API Provider set to "Mistral" and Model set to codestral-latest](figures/cline-setup-mistral-provider.png)

— [Cline docs, Mistral provider setup](https://docs.cline.bot/provider-config-mistral-ai)

Mistral's free "Experiment" tier gives rate-limited access at no cost. This is the key we'll use in the guided practice below.

### Setup 3 — a backup free key: Google Gemini

1. Go to Google AI Studio. Sign in with your Google account.
2. Navigate to [aistudio.google.com/apikey](https://aistudio.google.com/apikey).
3. Click "Create API Key" and select or create a Google Cloud project.
4. Copy the API key immediately and store it securely.

Then, in Cline:

1. Click the settings icon (⚙️) in the Cline panel.
2. Choose "Google Gemini" from the "API Provider" dropdown.
3. Paste your Google AI API key into the "Gemini API Key" field.
4. Choose `gemini-2.5-flash` from the "Model" dropdown.

![placeholder: Cline's settings panel with API Provider set to "Google Gemini" and Model set to gemini-2.5-flash](figures/cline-setup-gemini-provider.png)

— [Cline docs, Google Gemini provider setup](https://docs.cline.bot/provider-config/google-gemini)

No credit card is required for a Gemini API key. **Configure this now too**, even though the exercises only need one provider: some institutional Google accounts can't create an AI Studio key, and having a second provider ready means hitting a rate limit mid-exercise is a 10-second switch in Cline's settings, not a dead end. Check current free-tier limits at the [Gemini rate limits page](https://ai.google.dev/gemini-api/docs/rate-limits) or [Mistral's pricing page](https://mistral.ai/pricing) — don't trust a number printed in this notebook, they move.

### Where keys live

Cline stores your provider keys in VS Code's secret storage — not in this repository. For `5_generative_models.ipynb` and `rag_app/`, the course convention is a local `.env` file loaded with `python-dotenv`; `.env` is already listed in `.gitignore` so it never gets committed.

Three rules, no exceptions:
1. Never paste a key into a notebook cell.
2. Never paste a key into a chat message to an agent.
3. Never commit a key, even in a "temporary" file.

If a key ever leaks (pushed by mistake, pasted into a shared chat), rotate it immediately from the provider's dashboard — a leaked free-tier key is someone else's free API access, at your expense.


In [ ]:
from pathlib import Path
import os

gitignore = Path(".gitignore").read_text() if Path(".gitignore").exists() else ""
print("'.env' listed in .gitignore:", "PASS" if ".env" in gitignore else "FAIL")

env_file = Path(".env")
print(".env file present locally:", env_file.exists())

for key_name in ["MISTRAL_API_KEY", "GEMINI_API_KEY"]:
    print(f"{key_name} set in environment:", key_name in os.environ)


### Where the tools differ — a recap

| | Cline | Claude Code | Codex |
|---|---|---|---|
| Interface | VS Code extension (also Roo Code fork) | Terminal, IDE extension, desktop app | Terminal, ChatGPT app |
| Cost & auth | Free — bring your own API key (Codestral, Gemini, others) | Paid subscription or API key | Paid subscription or API key |
| Plan / Act split | Plan mode / Act mode | `plan` mode | plan-first workflow |
| Rules file | `.clinerules/` (also reads `AGENTS.md`, `.cursorrules`) | `./CLAUDE.md` | `AGENTS.md` |
| Approval granularity | Auto Approve per action type, or YOLO mode | Allow/Ask/Deny per tool | tool-approval settings |
| Skills / commands | Slash commands / workflows | Skills (`SKILL.md`) + slash commands, e.g. `/code-review` | custom prompts |
| Hooks | — | Yes | — |
| Multi-agent | — | Subagents: "spawn multiple Claude Code agents that work on different parts of a task simultaneously" ([docs](https://code.claude.com/docs/en/overview)) | — |

## Part 3 — Guided practice (~30 min)

Work in pairs, one laptop per pair, Cline configured with at least one free provider key.

**Setup (2 min).** Don't edit this notebook directly — work on a copy:
1. In VS Code's Explorer, duplicate this file (`0_coding_agents.ipynb`) and rename the copy, e.g. `0_coding_agents_practice.ipynb`.
2. Stage the copy so you have a clean baseline to diff and revert against: `git add 0_coding_agents_practice.ipynb`.
3. Open the copy — that's what you'll ask the agent to edit for the rest of this section.

**Exercise 1 (10 min) — Plan first.**
1. In Cline's **Plan** mode, ask it to add **one new markdown cell and one new code cell at the very end** of your copy, explaining one concept from Part 1 (your choice — e.g. the difference between a skill and a hook) in its own words, with a small runnable example. Explicitly tell it not to touch any existing cell.
2. Read the proposed plan against the five-point checklist above *before* switching to Act. Does the explanation match what was actually taught above? Does the plan touch anything beyond the two new cells you asked for?

**Exercise 2 (10 min) — Act and inspect.**
1. Switch to **Act**, approve, and let it make the change.
2. Read the actual diff: `git diff 0_coding_agents_practice.ipynb` (or VS Code's Source Control view). Confirm nothing above your new cells changed.
3. Run the new code cell — does it actually work?
4. `git checkout -- 0_coding_agents_practice.ipynb` to revert back to your clean copy — confirming the undo habit works before you rely on it for real.

**Exercise 3 (10 min, stretch) — write a rule.**
1. Create a throwaway `.clinerules/notebook-safety.md` with three lines: "never rewrite or restructure existing cells unless explicitly asked", "keep added comments to one line", "explain your reasoning before editing".
2. Re-run Exercise 1's request in a fresh task. What changed in the plan it proposed?
3. Delete the file (or leave it uncommitted) when done — it was only for practice.

### Debrief (~5 min)

Discuss with your pair:

1. Which mode would you default to for a genuinely one-line change — Plan or straight to Act? Why?
2. What did the rules file in Exercise 3 actually change about the agent's behavior?
3. Name one thing the agent did during these exercises that you could *not* have explained if asked in an exam.


## Checklist before day 6

Before day 6, make sure you have:

* [ ] VS Code installed, with Cline installed as an extension.
* [ ] At least one free provider key configured in Cline (Codestral and/or Gemini); ideally both, so a rate limit isn't a dead end.
* [ ] This repository cloned locally and opened **as a folder** in VS Code.
* [ ] The Python environment from `requirements.txt` set up and working.
* [ ] Completed one full Plan → Act → read-the-diff → revert cycle on your own (repeat Exercises 1–2 above).
* [ ] You know where your keys live (VS Code secret storage / a local `.env`) and that `.env` is never committed.
* [ ] Can explain, in your own words, what a skill and a hook each do differently.

Claude Code and/or Codex, installed and logged in, are welcome additions if you already pay for them — but nothing above requires either.


## References

**Claude Code**
- [Overview](https://code.claude.com/docs/en/overview)
- [Memory](https://code.claude.com/docs/en/memory)
- [Configure permissions](https://code.claude.com/docs/en/permissions)
- [Model configuration](https://code.claude.com/docs/en/model-config)
- [Commands reference](https://code.claude.com/docs/en/commands)
- [Code Review](https://code.claude.com/docs/en/code-review)
- [Skills](https://code.claude.com/docs/en/skills)
- [Hooks guide](https://code.claude.com/docs/en/hooks-guide)

**Cline**
- [Installing Cline](https://docs.cline.bot/getting-started/installing-cline)
- [Mistral / Codestral provider setup](https://docs.cline.bot/provider-config/mistral-ai)
- [Google Gemini provider setup](https://docs.cline.bot/provider-config/google-gemini)
- [Plan & Act](https://docs.cline.bot/features/plan-and-act)
- [Auto Approve](https://docs.cline.bot/features/auto-approve)
- [Cline Rules](https://docs.cline.bot/customization/cline-rules)
- [Slash Commands](https://docs.cline.bot/core-workflows/using-commands)

**General code review**
- Google Engineering Practices, [What to look for in a code review](https://google.github.io/eng-practices/review/reviewer/looking-for.html)

**Codex**
- [Codex docs](https://learn.chatgpt.com/docs)

**Free-tier limits (check current numbers here, not in this notebook)**
- [Gemini API rate limits](https://ai.google.dev/gemini-api/docs/rate-limits)
- [Mistral AI pricing](https://mistral.ai/pricing)
